In [ ]:
from mayavi import mlab
import pandas as pd
import numpy as np
import os, random, string
import nibabel as nb
from matplotlib import pyplot as plt
import matplotlib
from scipy.ndimage import distance_transform_edt
from matplotlib.colors import ListedColormap
import matplotlib.colors as mcolors
from matplotlib.ticker import FuncFormatter
from mpl_toolkits.mplot3d import Axes3D
from skimage.measure import label, regionprops
from matplotlib.colors import to_rgb

from tqdm import trange

In [ ]:
def load_nifti(src: str):
    """
    Load nifti image. Array is rearranged so that it aligns with TIFF or HDF5 format.
    Parameters
    ----------
    src : str
    """
    stack = nb.load(src).get_fdata()
    # the below line is needed to make it consistent with TIFF array order
    stack = np.swapaxes(stack, 0, 2)
    return stack

In [ ]:
atlas_path='/bucket/ReiterU/DBS/atlas/S_lessoniana_OL_v0.4/segmentation/atlas_seg10.nii.gz'
atlas=load_nifti(atlas_path)
masked_atlas=np.copy(atlas)
dt1,dt1_indexes = distance_transform_edt(atlas != 3.,return_indices=True) # compute the atlas as distance from the OGL (superficial layer of the OL).
masked_atlas[masked_atlas==0]=np.nan


In [ ]:
with open('EXP_NAMES.txt', 'r') as f:
    EXP_NAMES = [line.strip() for line in f if line.strip()]
gen_path='/bucket/ReiterU/electrophysiology/squidOL/'


In [ ]:
### Get a list of the correct recording names for each experiment ###
# STAB_CHECK is the standard initial recording, with no stimulus
rec_paths=[]
for exp_name in EXP_NAMES:
    rec_list=os.listdir(gen_path+exp_name)
    found=False
    for i in rec_list:
        if 'STAB_CHECK' in i:
            rec_paths.append(gen_path+exp_name+'/'+i)
            found=True
            break
    if found==False:
        print('not found in {}'.format(exp_name))

In [ ]:
dataset_ids=[]
ALL_spikes=[]
ALL_ids=[]
ALL_channels=[]
probe_data=[]
for rec in trange(len(rec_paths)):
    rec_path = rec_paths[rec]
    exp_path = gen_path + EXP_NAMES[rec]
    post_path = rec_path + '/post_processed/'
    for probe in os.listdir(post_path):
        if len(os.listdir(post_path + '/' + probe)) != 0:
            sort_path = post_path + '/' + probe + '/'
            dataset_ids.append(EXP_NAMES[rec] + '_' + probe)
            spike_t_matrix = np.load(sort_path + '/spike_t_matrix.npy', allow_pickle=True)
            g_channels = np.load(sort_path + '/g_channels.npy')
            g_ids = np.load(sort_path + '/g_ids.npy')
            ALL_spikes.append(spike_t_matrix)
            ALL_ids.append(g_ids)
            ALL_channels.append(g_channels)

            if probe == 'probe1':
                probe_path = exp_path + '/probe_skeleton/probe_2_transformed.csv'
            elif probe == 'probe0':
                probe_path = exp_path + '/probe_skeleton/probe_1_transformed.csv'
            probe_data.append(probe_path)




In [ ]:
better_hex = [
    "#e6194b", "#3cb44b", "#ffe119", "#4363d8", "#f58231", "#911eb4", "#46f0f0",
    "#f032e6", "#bcf60c", "#fabebe", "#008080", "#e6beff", "#9a6324", "#fffac8",
    "#800000", "#aaffc3", "#808000", "#ffd8b1", "#000075", "#808080", "#ffffff",
    "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2",
    "#7f7f7f", "#bcbd22", "#17becf", "#aec7e8", "#ffbb78", "#98df8a", "#ff9896",
    "#c5b0d5", "#c49c94", "#f7b6d2", "#c7c7c7", "#dbdb8d", "#9edae5",
    "#332288", "#88CCEE", "#117733", "#DDCC77", "#CC6677", "#AA4499", "#44AA99", "#999933",
    "#1b9e77", "#d95f02", "#7570b3", "#e7298a", "#66a61e", "#e6ab02", "#a6761d", "#666666",
    "#4daf4a", "#984ea3", "#377eb8", "#ff7f00", "#ffff33", "#a65628", "#f781bf", "#999999",
    "#006400", "#8b0000", "#483d8b", "#008b8b", "#b22222","#aaffc3", "#808000", "#ffd8b1", "#000075"
]

# Load and process probe data
probes = []
probe_chan_regions = []
names = []

for d in probe_data:
    df = pd.read_csv(d)
    df = df.loc[50:]  # crop extrapolated channels
    zyx = df[["Z", "Y", "X"]].to_numpy()
    chan_reg = df[['Region']].to_numpy()
    probes.append(zyx)
    probe_chan_regions.append(chan_reg)

# Scale and convert probe coordinates to match atlas index scale
probes_smooth = [(np.asarray(probe) / 10).astype(int) for probe in probes]  # /10 to match the atlas

# Convert hex to RGB
probe_colors = [to_rgb(h) for h in better_hex[:len(probes)]]



In [ ]:
# Append coordinates to list of matrices. The shape matches all previous created lists : Number_of_probes X number_of_units_on_probe
ALL_UNIT_COORDS=[] # Real coordinates
ALL_UNIT_SMOOTH_COORDS=[] # Atlas index coordinates
for data_index in range(len(dataset_ids)):
    UNIT_COORDS=[]
    UNIT_SMOOTH_COORDS=[]
    g_ids=ALL_ids[data_index]
    g_channels=ALL_channels[data_index]

    for unit in range(len(g_ids)):
        unit_channel=g_channels[unit]
        chan_coords=probes[data_index][unit_channel]
        chan_smooth_coords=probes_smooth[data_index][unit_channel]
        UNIT_COORDS.append(chan_coords)
        UNIT_SMOOTH_COORDS.append(chan_smooth_coords)
    ALL_UNIT_COORDS.append(np.asarray(UNIT_COORDS))
    ALL_UNIT_SMOOTH_COORDS.append(np.asarray(UNIT_SMOOTH_COORDS))

In [219]:
#Snip of code to plot the Atlas with appended probes used in the paper

mlab.figure(size=(150,150), bgcolor=(1, 1, 1))

regions = ["IGL", "MED", "OGL", "PD", "PXL","CB"]
colors = (
    (0.96,0.50,0.10),
    (0.50,0.18,0.70),
    (0.031, 0.188, 0.420),
    (0.35,0.45,1.00),
    (0.2,0.8,0.2),
    (0.90,0.07,0.04)
)
for r, c in zip(regions, colors):
    d = mlab.pipeline.open(f"/bucket/ReiterU/DBS/atlas/S_lessoniana_OL_v0.4/segmentation/{r}.ply")
    mlab.pipeline.surface(d, color=c, opacity=0.05)

for i, p in enumerate(probes):
    mlab.plot3d(p[:,2], p[:,1], p[:,0], color=probe_colors[i], tube_radius=15)
mlab.view(180, 180, 10000, focalpoint=(1500, 2500, 1000)) # coronal
#mlab.view(0, 90, 10000, focalpoint=(1500, 2200, 2000), roll=180) # sagittal
#mlab.view(90, 270, 10000, focalpoint=(1500, 2200, 1500), roll=0) # horizontal
mlab.savefig(save_path+'ALL_PROBES_ATLAS_FIG_CORONAL.png',magnification=8)
mlab.close(all=True)
#mlab.show()


In [221]:
#Snip of code to plot the Atlas with appended unit locations used in the paper

mlab.figure(size=(150,150), bgcolor=(1, 1, 1))

regions = ["IGL", "MED", "OGL", "PD", "PXL", "CB"]
colors = (
    (0.96,0.50,0.10),
    (0.50,0.18,0.70),
    (0.031, 0.188, 0.420),
    (0.35,0.45,1.00),
    (0.2,0.8,0.2),
    (0.90,0.07,0.04)
)
for r, c in zip(regions, colors):
    d = mlab.pipeline.open(f"/bucket/ReiterU/DBS/atlas/S_lessoniana_OL_v0.4/segmentation/{r}.ply")
    mlab.pipeline.surface(d, color=c, opacity=0.05)

for data_index in range(len(dataset_ids)):
    UNIT_COORDS = ALL_UNIT_COORDS[data_index]
    mlab.points3d(UNIT_COORDS[:,2], UNIT_COORDS[:,1], UNIT_COORDS[:,0], color=(0,0,0), scale_factor=50)

#mlab.view(180, 180, 10000, focalpoint=(1500, 2500, 1000)) # coronal
mlab.view(0, 90, 10000, focalpoint=(1500, 2200, 2000), roll=180) # sagittal
#mlab.view(90, 270, 10000, focalpoint=(1500, 2200, 1500), roll=0) # horizontal
mlab.savefig(save_path + "All_UNIT_SAG.png", magnification=8)
#mlab.show()
mlab.close(all=True)


In [ ]:
segs=np.unique(atlas)
ALL_UNIT_DISTS=[]
ALL_UNIT_SEGS=[]
for data_index in range(len(dataset_ids)):
    UNIT_DIST=[]
    UNIT_SEGS=[]
    g_ids=ALL_ids[data_index]
    for unit in range(len(g_ids)):
        z,y,x=ALL_UNIT_SMOOTH_COORDS[data_index][unit][0],ALL_UNIT_SMOOTH_COORDS[data_index][unit][1],ALL_UNIT_SMOOTH_COORDS[data_index][unit][2]
        seg=masked_atlas[z][y][x]
        unit_dist=dt1[z][y][x]
        UNIT_DIST.append(unit_dist)
        UNIT_SEGS.append(seg)

    ALL_UNIT_DISTS.append(np.asarray(UNIT_DIST))
    ALL_UNIT_SEGS.append(np.asarray(UNIT_SEGS))
ALL_UNIT_DISTS=np.asarray(ALL_UNIT_DISTS,dtype='object')
ALL_UNIT_SEGS=np.asarray(ALL_UNIT_SEGS,dtype='object')

In [ ]:
SEG_LABELS=['MEDULLA','PED','OGL','PLEX','IGL','CB']
SEG_COLORS=['#FF7F0E','#3a001e','#9467BD','#a72f65','#17BECF','#FF2F72']
good_colors=SEG_COLORS
good_colors=ListedColormap(good_colors)

In [ ]:
# Example coronal atlas slice 
width=9.2/2.54
height=12/2.54
plt.figure(figsize=(5*cm,5*cm))
plt.imshow(masked_atlas[220],cmap=good_colors,alpha=0.95,interpolation='nearest')
plt.axis('off')
plt.savefig(save_path+'ATLAS_SEG_CORONAL_SECTION.svg',format='svg',dpi=600)
plt.show()
#plt.close()

In [ ]:
dist_probes=[]
for i,probe in enumerate(probes_smooth):
    dist_probe=[]
    for j in probe:
        dist=dt1[j[0],j[1],j[2]]
        dist_probe.append(dist)
    dist_probes.append(dist_probe)

In [ ]:
# Distance over depth for the channels from each probe

plt.figure(figsize=(5/2.54,4/2.54))

# Reverse each vector in dist_probes before plotting
for j,i in enumerate(dist_probes):
    plt.plot(i[::-1], alpha=0.7,linewidth=2,color=better_hex[j]) 

# Customize plot appearance
ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.xlabel('channels')
plt.ylabel('depth (mm)')
plt.yticks([0,50,100,150],[0,0.5,1,1.5])
plt.xlim([0,400])
plt.ylim([0,150])
# Show the plot
plt.tight_layout()
plt.savefig(save_path+'distance_over_probes.svg',format='svg',dpi=600)
plt.show()
#plt.close()

In [ ]:
flat_UNIT_DISTS=np.concatenate(ALL_UNIT_DISTS)

In [ ]:
bin_size=20
bins=np.arange(0,np.nanmax(flat_UNIT_DISTS)+bin_size,bin_size)
udd,_=np.histogram(flat_UNIT_DISTS+10,bins)

In [ ]:
# Units distributed across depth bins

plt.figure(figsize=(5*cm,4*cm))
ax=plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.bar(bins[:-1],udd,width=bin_size-(bin_size)/10,color='black')
plt.ylabel('Num. Units')
plt.xlabel('depth (mm)')
plt.xticks([0,40,80,120],[0,0.4,0.8,1.2])
plt.yticks([0,250,500,750])
plt.tight_layout()
plt.savefig(save_path+'unit_distance_distribution.svg',format='svg',dpi=600)
plt.show()
#plt.close()

In [ ]:
# Same atlas slice as before, displaying the depth gradient from OGL

atlas_index = 220
bin_range = np.arange(0, 140, 10)

# Prepare binned data with NaNs
nan_mask = np.isnan(dt1[atlas_index])
binned_data = np.digitize(dt1[atlas_index], bin_range).astype(float)
binned_data[nan_mask] = np.nan
binned_data[binned_data > 1] += 1

# Create figure and axis
fig, ax = plt.subplots(figsize=(5/2.54, 5/2.54))

# Plot contours from the masked atlas
ax.contour(masked_atlas[atlas_index], 
           levels=np.unique(masked_atlas[atlas_index]), 
           colors='black', linewidths=0.5)

# Overlay the binned data with transparency
im = ax.imshow(binned_data, cmap='viridis', alpha=0.8)
cbar_ax = fig.add_axes([0.78, 0.16, 0.03, 0.4])  # adjust as needed
cbar = fig.colorbar(im, cax=cbar_ax)
cbar.set_label('depth (mm)')
cbar.set_ticks([1, 5, 9, 13])
cbar.set_ticklabels([0, 0.4, 0.8, 1.2])
ax.axis('off')
plt.savefig(save_path+'checking_depth_overlay_only.svg',format='svg', bbox_inches='tight', dpi=600)
plt.show()
#plt.close()